# 氣象資料改用竹東測站：分析結果變化說明

2026-10-20

## 摘要

- 氣象資料由中央氣象署**新竹站（467571）**改為**竹東站（C0D560）**，所有分析與模型已用新資料重新計算。
- **主機本身的結果不受影響**：運轉狀況、性能曲線都不使用氣象資料。
- **主要變化在出水溫度調高的節電估計**：每調高 1°C，資料估計從約 1.2% 提高到約 1.7%（理論上限 2.8% 不變）。
- **建議不變**：出水溫度仍建議先試行，用實測確認節電量。只換測站就讓節電估計改變約 1.5 倍，說明在實際調整出水溫度之前，現有資料無法定出精確的節電量。

## 1. 兩個測站有什麼不同

分析期間 2026/09/17 ~ 10/13 的逐時觀測比較：

| 項目 | 新竹站 | 竹東站 |
|---|---|---|
| 平均溫度 | 27.2°C | 27.0°C |
| 每日最高溫（平均） | 31.1°C | 32.1°C |
| 每日最低溫（平均） | 24.1°C | 23.2°C |
| 每日最高濕球溫度（平均） | 24.8°C | 25.2°C |
| 平均氣壓 | 1010.6 hPa | 997.9 hPa |

- **平均值幾乎相同**：溫度、濕球溫度、外氣焓值的平均差都在 0.3 以內，逐時變化的相關係數 0.96 以上
- **竹東站日夜溫差較大**：白天較熱、夜間較涼。冰水主機在白天運轉，因此模型看到的「外氣與負荷、耗電的關係」會跟著改變
- **氣壓較低**：兩站高度不同，屬正常差異，不是資料錯誤

**資料處理方式**
- 竹東站沒有露點觀測，改由溫度、濕度換算
- 竹東站沒有日射量、雲量、能見度，這些欄位不再用於模型
- 即時預測用的天氣預報地點，同步由新竹市東區改為新竹縣竹東鎮

## 2. 各項分析的變化

### 2.1 CH-2 出水溫度調高的節電估計（`ch2_power_model`）—— 影響最大

| 項目 | 新竹站 | 竹東站 |
|---|---|---|
| 耗電模型準確度 CV(RMSE)（越低越好，標準 < 30%） | 3.4% | 3.5% |
| 調高 0.5°C 節電（資料估計） | 0.6% | 0.9% |
| 調高 1.0°C 節電（資料估計） | 1.2% | 1.7% |
| 調高 1.5°C 節電（資料估計，外推） | 1.7% | 2.6% |
| 調高 1.0°C 的 95% 區間 | −0.05% ~ 6.0% | 0.6% ~ 6.5% |
| 調高 1.0°C 節電（理論上限） | 2.8% | 2.8% |

- 採用的模型不變（灰箱線性，外氣用濕球），準確度相當
- 資料估計的節電約變為原本的 1.5 倍；「調高會省電」的方向不變，也仍低於理論上限
- **為什麼會變**：濕熱的時段負荷大，出水溫度會偏高，主機也本來就比較耗電。模型用外氣濕球扣掉這部分耗電，才算得出出水溫度的省電效果。竹東站白天較濕熱（每日最高濕球 25.2°C，新竹站 24.8°C），用竹東站資料時，模型判定濕熱造成的耗電較多（濕球每高 1°C 多耗 0.77 kW，新竹站 0.31 kW），扣掉較多，節電估計就較高
- **誤差沒有比較大**：兩站模型準確度相當（CV(RMSE) 3.4% / 3.5%），95% 區間寬度也相近
- **對決策的意義**：調高 0.5~1°C 的預估節電範圍由 0.6%~2.8% 變為 0.9%~2.8%。範圍的上限不變，下限提高；實際數字仍需試行確認

### 2.2 主機與外氣的關聯（`chiller_analysis`）

| 指標（R²，越接近 1 關係越強） | 新竹站 | 竹東站 |
|---|---|---|
| 冷凍噸 vs 外氣濕球 | 0.77 | 0.71 |
| 冷凍噸 vs 外氣乾球 | 0.38 | 0.34 |
| 耗電 vs 外氣焓值 | 0.74 | 0.69 |
| 出水溫度 vs 外氣乾球 | 0.05 | 0.03 |

- 關聯略為下降，但結論不變：濕球溫度最能解釋負荷；出水溫度設定固定，不隨外氣調整
- 這些 R² 只代表外氣與負荷、耗電的關聯強弱，不代表節電估計的誤差

### 2.3 CH-2 候選模型比較（`ch2_power_model`）

| 模型 | 準確度 CV(RMSE)<br>新竹站 | 準確度 CV(RMSE)<br>竹東站 | 每調高 1°C 節電<br>新竹站 | 每調高 1°C 節電<br>竹東站 |
|---|---|---|---|---|
| 灰箱線性（濕球，採用） | 3.4% | 3.5% | 1.73% | 2.57% |
| 灰箱線性（乾球） | 3.9% | 3.4% | 1.76% | 1.97% |
| 灰箱線性（乾球 + 濕球） | 4.3% | 3.6% | 1.84% | 2.51% |
| 灰箱線性（全部氣象） | 3.8% | 3.8% | 1.91% | 3.05% |
| LightGBM（3 種） | 11.2~11.6% | 12.1~12.2% | 調高反而更耗電或幾乎沒反應 | 同左 |

- 採用的灰箱線性（濕球）在兩站準確度都是最好或接近最好；冷卻水塔由濕球決定，兩站都採用它
- 新竹站各灰箱模型的節電估計很接近（1.7~1.9%），竹東站差距較大（2.0~3.1%）：節電估計受外氣資料影響大
- 本表只算穩定運轉、未負荷過大的時段；2.1 含不調整的時段，所以數字較低